# Chapter 02 · Mathematics: Batch averages and update counts

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: optional reference for later; not yet reviewed.** The programming lessons are already covered. Continue the programming-led mentorship without needing to complete this companion first. Use small numerical explanations when helpful, and revisit formal notation gradually.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Chapter 01 math: mean squared error and gradients.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. Average gradient contributions in a batch](#batch-mean)
2. [2. Why sequential updates depend on order](#order)
3. [3. Count batches, updates, and epochs](#counts)

## Setup

In [ ]:
import torch

<a id="batch-mean"></a>
## 1. Average gradient contributions in a batch


A batch is a group of examples. Let $\mathcal B$ mean that group and $m$ mean its actual number of examples. For our single-output model:

$$L_{\mathcal B}=\frac{1}{m}\sum_{i\in\mathcal B}(wx_i+b-y_i)^2$$

The notation $i\in\mathcal B$ means “use the examples belonging to this batch.” Its gradient is the average of their individual contributions, **all evaluated at the same current parameter values**:

$$g_{w,\mathcal B}=\frac{1}{m}\sum_{i\in\mathcal B}2e_ix_i,\qquad g_{b,\mathcal B}=\frac{1}{m}\sum_{i\in\mathcal B}2e_i$$

For weight `1`, bias `0`, examples `(x=0, y=2)` and `(x=1, y=5)` have individual gradients `(0, -4)` and `(-8, -8)` for `(weight, bias)`. Their batch gradient is `(-4, -6)`.

A mean gradient does not normally grow just because the batch contains more examples; it reflects the average contribution. The included examples still affect that average.

**Connection:** programming Lessons 01 and 03. **Before running:** calculate the average of the two weight gradients and the two bias gradients separately.


**My prediction or explanation:** _write here_

In [ ]:
# Calculate individual contributions at one fixed parameter setting.
x_batch = torch.tensor([0., 1.])
y_batch = torch.tensor([2., 5.])
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
errors = weight * x_batch + bias - y_batch
weight_contributions = 2 * errors.detach() * x_batch
bias_contributions = 2 * errors.detach()
batch_loss = (errors ** 2).mean()
batch_loss.backward()
print("Individual weight contributions:", weight_contributions.tolist())
print("Individual bias contributions:", bias_contributions.tolist())
print("Mean weight gradient:", weight.grad.item())
print("Mean bias gradient:", bias.grad.item())

<a id="order"></a>
## 2. Why sequential updates depend on order


Let $\theta$ (theta) be shorthand for all the parameters, here the pair `(weight, bias)`. Let $g_A(\theta)$ mean the gradient on example A evaluated at parameters $\theta$.

Two sequential steps follow this pattern:

$$\theta_1=\theta_0-\eta g_A(\theta_0),\qquad \theta_2=\theta_1-\eta g_B(\theta_1)$$

The second gradient is calculated at the already changed parameters $\theta_1$. Reversing A and B changes what the second step sees. It is not the same operation as averaging two gradients at the original parameters and taking one step.

**Connection:** programming Lessons 01–03. **Before running:** should reversing two examples always produce identical parameters after two sequential updates?


**My prediction or explanation:** _write here_

In [ ]:
# Use the same examples, initial values, learning rate, and two-update budget.
def two_sequential_updates(examples):
    weight, bias = 1.0, 0.0
    for input_value, target in examples:
        error = weight * input_value + bias - target
        gradient_weight = 2 * error * input_value
        gradient_bias = 2 * error
        # Both gradients use the same pre-update parameter values.
        weight -= 0.05 * gradient_weight
        bias -= 0.05 * gradient_bias
    return weight, bias

examples = [(0.0, 2.0), (1.0, 5.0)]
print("Order A then B:", two_sequential_updates(examples))
print("Order B then A:", two_sequential_updates(list(reversed(examples))))

<a id="counts"></a>
## 3. Count batches, updates, and epochs


An epoch is one complete loader pass. In our loops we call `optimizer.step()` once per batch. When $N$ examples divide evenly into batches of $B$ examples, the counts are:

$$\text{updates per epoch}=N/B,\qquad \text{total updates}=E(N/B)$$

$N$ is dataset size, $B$ is batch size, and $E$ is the number of epochs. Four examples, batch size two, and two epochs give `2` batches per epoch and `4` total updates.

For four examples, batch sizes `1`, `2`, and `4` give respectively `4`, `2`, and `1` updates per epoch. Thus “one epoch” is not a fixed update budget when comparing batch sizes. If a final batch is incomplete, keeping or skipping it changes the count; Chapter 06's math companion explains the rounding.

**Connection:** programming Lessons 04–05. **Before running:** how many calls to `step()` occur over three epochs with four examples and batch size two?


**My prediction or explanation:** _write here_

In [ ]:
# This count assumes one optimizer step per batch and no skipped examples.
number_of_examples, batch_size, epochs = 4, 2, 3
batches_per_epoch = number_of_examples // batch_size
print("Updates per epoch:", batches_per_epoch)
print("Total updates:", epochs * batches_per_epoch)
print("Example visits:", epochs * number_of_examples)

**My observation:** _write here_

Distinguish the number of examples, the number of batches, and the number of optimizer steps. Average gradients at one fixed parameter setting; do not confuse that with averaging sequential training results.